# Week 14 Lab — Layers, rivers, and inheriting a model

**HWRS 564a · Fall 2026**

Two things left, and they are the two you are most likely to need in practice.

**Layers**, because aquifers are not one thing. A confining unit between two
sands changes where water comes from and how fast it arrives, and a single-layer
model cannot represent it at all.

**Loading somebody else's model**, because that is how most groundwater modelling
actually starts. You will almost never build from nothing — you will be handed a
directory of input files by a consultant, an agency, or a graduate student who
left.

One session this week; Thursday is Thanksgiving.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Build a multi-layer model, and set vertical conductivity deliberately
2. Read a vertical head difference and say what it means
3. Add a river with the `RIV` package and interpret its budget term
4. Load an existing model from its `.nam` file with `Modflow.load()`
5. Modify a loaded model, re-run it, and compare the two results
6. Say why comparing two models is more defensible than trusting one

---

## Part 1 — Setup

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
MF_EXE = ROOT / "modflow" / "mf2005"
assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Run ./postbuild.sh from a terminal."
)

RUN = ROOT / "_run"
NROW, NCOL, DELR = 40, 60, 250.0
UA_RED, UA_BLUE, UA_SKY = "#AB0520", "#0C234B", "#81D3EB"


def discrepancy(list_path):
    for line in reversed(Path(list_path).read_text().splitlines()):
        if "PERCENT DISCREPANCY" in line:
            return float(line.split()[-1])
    raise ValueError(f"no budget discrepancy in {list_path}")


def budget_terms(list_path):
    lines = Path(list_path).read_text().splitlines()
    start = max(i for i, l in enumerate(lines) if "VOLUMETRIC BUDGET" in l)
    half, terms = "in", {}
    for line in lines[start:start + 32]:
        t = line.strip()
        if t.startswith("OUT:"):
            half = "out"
        if "=" not in t or t.startswith(("TOTAL", "IN - OUT", "PERCENT")):
            continue
        terms.setdefault(t.split("=")[0].strip(), {})[half] = float(t.split()[-1])
    return terms


print("ready")

---

## Part 2 — Three layers

Our basin fill is really two sand units separated by a clay. Modelled as three
layers:

| Layer | Elevation | Material | `hk` (m/d) |
|---|---|---|---|
| 1 | 600 – 520 m | upper basin fill | 15 |
| 2 | 520 – 480 m | **clay confining unit** | 0.05 |
| 3 | 480 – 400 m | lower basin fill | 25 |

`botm` takes a list — one bottom elevation per layer. `top` is still a single
surface: the top of layer 1.

In [ ]:
BOTM = [520.0, 480.0, 400.0]
AQ_TOP = 600.0
NLAY = len(BOTM)

WS = RUN / "week14_layered"
WS.mkdir(parents=True, exist_ok=True)

mf = flopy.modflow.Modflow("layered", model_ws=str(WS), exe_name=str(MF_EXE))
flopy.modflow.ModflowDis(
    mf, NLAY, NROW, NCOL, delr=DELR, delc=DELR,
    top=AQ_TOP, botm=BOTM, nper=1, steady=True,
)

thicknesses = [AQ_TOP - BOTM[0]] + [BOTM[i - 1] - BOTM[i] for i in range(1, NLAY)]
for i, (t, b) in enumerate(zip(thicknesses, BOTM)):
    print(f"layer {i}: {t:5.0f} m thick, bottom at {b:.0f} m")

### Horizontal and vertical conductivity are different arguments

`hk` is horizontal; `vka` is vertical. In layered sediments they are not the
same — deposition makes horizontal flow easier than vertical, typically by a
factor of 10 to 100. That ratio is the **anisotropy**, and it is what makes a
confining unit confine.

In [ ]:
hk = np.zeros((NLAY, NROW, NCOL))
hk[0] = 15.0     # upper sand
hk[1] = 0.05     # clay
hk[2] = 25.0     # lower sand

vka = hk * 0.1   # vertical K is a tenth of horizontal

flopy.modflow.ModflowLpf(mf, hk=hk, vka=vka, laytyp=0, ipakcb=53)

print("layer   hk (m/d)   vka (m/d)   T (m2/d)")
for i in range(NLAY):
    print(f"  {i}   {hk[i, 0, 0]:8.2f}   {vka[i, 0, 0]:9.3f}   "
          f"{hk[i, 0, 0] * thicknesses[i]:8.0f}")

### YOUR TURN 1

The clay's job is to resist vertical flow. Quantify how well it does it.

**Vertical leakance** between two layers is the conductance per unit area:

$$\frac{1}{\text{leakance}} = \frac{b_1 / 2}{K_{v1}} + \frac{b_2 / 2}{K_{v2}}$$

— half the thickness of each layer divided by its vertical conductivity, added in
series like electrical resistances.

- `resistance_clay` — vertical resistance of the clay layer alone
  (its full thickness divided by its `vka`)
- `resistance_upper_sand` — the same for the upper sand
- `resistance_ratio` — how many times more resistant the clay is

In [ ]:
# YOUR TURN
resistance_clay = ...
resistance_upper_sand = ...
resistance_ratio = ...

In [ ]:
# CHECK
assert abs(resistance_clay - 8000.0) < 1.0, f"got {resistance_clay}"
assert abs(resistance_upper_sand - 53.33) < 0.5, f"got {resistance_upper_sand}"
assert abs(resistance_ratio - 150.0) < 1.0, f"got {resistance_ratio}"
print(f"clay resistance        {resistance_clay:9,.0f} days")
print(f"upper sand resistance  {resistance_upper_sand:9,.1f} days")
print(f"ratio                  {resistance_ratio:9,.0f}x")
print("\nThe clay is only 40 m thick and provides 150x the resistance of the")
print("80 m sand above it. Correct.")

**That is what a confining unit is** — not an impermeable barrier, but a layer
whose resistance dominates the vertical flow path. Water still crosses it; it
just takes a long time, and the head difference across it can be large.

---

## Part 3 — A river

The Santa Cruz runs down the middle of our domain. A river is not a constant
head: it exchanges water with the aquifer **through a streambed**, at a rate that
depends on the head difference and on how leaky the bed is.

`RIV` takes six values per cell: `[layer, row, col, stage, cond, rbot]`.

| Value | Meaning |
|---|---|
| `stage` | water-surface elevation in the river |
| `cond` | streambed conductance, m²/d — leakiness times area over thickness |
| `rbot` | elevation of the bottom of the streambed |

In [ ]:
RIVER_COL = 30
RIVER_STAGE = 665.0
RIVER_COND = 500.0        # m2/d per cell
RIVER_BOT = 660.0

river_spd = [[0, r, RIVER_COL, RIVER_STAGE, RIVER_COND, RIVER_BOT]
             for r in range(NROW)]
flopy.modflow.ModflowRiv(mf, stress_period_data={0: river_spd}, ipakcb=53)

print(f"river runs north-south down column {RIVER_COL}, {NROW} cells")
print(f"stage {RIVER_STAGE} m, bed bottom {RIVER_BOT} m, conductance {RIVER_COND} m2/d")

> **The rule that makes `RIV` different from a constant head.** If aquifer head
> is above `stage`, the aquifer drains to the river. If it is between `stage` and
> `rbot`, the river leaks into the aquifer at a rate proportional to the
> difference. And if head falls **below `rbot`**, the leakage rate stops
> increasing — the river is disconnected, and it cannot supply more water no
> matter how hard you pump.
>
> That last case is the one that matters. A constant-head boundary at the river
> would supply unlimited water forever, which is why using one to represent a
> river will make any pumping scenario look sustainable.

Now the rest of the model, and a well pumping from the **lower** aquifer,
underneath the clay.

In [ ]:
ibound = np.ones((NLAY, NROW, NCOL), dtype=int)
ibound[:, :, 0] = -1
ibound[:, :, -1] = -1
strt = np.full((NLAY, NROW, NCOL), 660.0)
strt[:, :, 0] = 620.0
strt[:, :, -1] = 700.0
flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)

WELL_LAY, WELL_ROW, WELL_COL = 2, 20, 35     # LOWER aquifer, east of the river
PUMP = -20000.0
flopy.modflow.ModflowWel(
    mf, stress_period_data={0: [[WELL_LAY, WELL_ROW, WELL_COL, PUMP]]}, ipakcb=53
)
flopy.modflow.ModflowPcg(mf)
flopy.modflow.ModflowOc(
    mf, stress_period_data={(0, 0): ["save head", "save budget", "print budget"]}
)

mf.write_input()
ok, buff = mf.run_model(silent=True, report=True)
assert ok, "MODFLOW did not converge:\n" + "\n".join(buff[-20:])

head = flopy.utils.HeadFile(str(WS / "layered.hds")).get_data()
print(f"converged, discrepancy {discrepancy(WS / 'layered.list'):+.3f} %")
print(f"head shape {head.shape}  (layers, rows, columns)")
for i in range(NLAY):
    print(f"  layer {i}: {head[i].min():.2f} to {head[i].max():.2f} m")

### YOUR TURN 2

The clay should hold a head difference between the two sands. Measure it at the
wellfield.

- `head_upper` — head in layer 0 above the well
- `head_lower` — head in layer 2 at the well
- `vertical_difference` — upper minus lower

In [ ]:
# YOUR TURN
head_upper = ...
head_lower = ...
vertical_difference = ...

In [ ]:
# CHECK
assert vertical_difference > 1.0, (
    f"the clay should sustain a head difference: got {vertical_difference:.2f} m"
)
print(f"layer 0 (upper sand)  {head_upper:8.2f} m")
print(f"layer 2 (lower sand)  {head_lower:8.2f} m")
print(f"difference across the clay {vertical_difference:6.2f} m")
print("\nCorrect. In a single-layer model this number does not exist.")

**Eight metres of head difference across 40 m of clay.** That downward gradient
is what drives leakage from the upper aquifer into the lower one — which is where
a good part of the well's water is coming from.

A single-layer model would have averaged all three units into one transmissivity
and reported a single head. It would still have balanced its budget, and it would
have had nothing to say about which aquifer the water came out of — which is
exactly the question a well-permit application asks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

for i, ax in enumerate(axes):
    im = ax.imshow(head[i], cmap="Blues_r", vmin=620, vmax=700,
                   extent=[0, NCOL * DELR / 1000, 0, NROW * DELR / 1000])
    ax.axvline(RIVER_COL * DELR / 1000, color=UA_SKY, lw=2.5)
    ax.set_title(f"layer {i}: {['upper sand', 'clay', 'lower sand'][i]}")
    ax.set_xlabel("distance east (km)")
axes[0].set_ylabel("distance north (km)")
axes[2].plot(WELL_COL * DELR / 1000, (NROW - WELL_ROW) * DELR / 1000,
             "*", color=UA_RED, ms=16)
fig.colorbar(im, ax=axes, label="head (m)", shrink=0.85)
plt.show()

The river shows as the pale line; the well as the star in layer 2. Note how much
smoother the clay layer's head field is — a low-conductivity layer averages over
its neighbours.

### YOUR TURN 3

Use the budget to find out where the well's water came from.

- `river_leakage_in` — water entering the aquifer from the river
- `river_drainage_out` — water leaving the aquifer to the river
- `net_river` — net river contribution to the aquifer

In [ ]:
terms = budget_terms(WS / "layered.list")
print("budget terms present:", sorted(terms))

# YOUR TURN
river_leakage_in = ...
river_drainage_out = ...
net_river = ...

In [ ]:
# CHECK
assert "RIVER LEAKAGE" in terms, f"no river term found: {sorted(terms)}"
assert abs(net_river - (river_leakage_in - river_drainage_out)) < 1e-6
print(f"river leakage into the aquifer  {river_leakage_in:10,.1f} m3/d")
print(f"aquifer drainage to the river   {river_drainage_out:10,.1f} m3/d")
print(f"net river contribution          {net_river:+10,.1f} m3/d")
print(f"as a share of pumping           {100 * net_river / -PUMP:+9.1f} %")
print("Correct.")

---

## Part 4 — Inheriting a model

`mf.write_input()` wrote a directory of text files. Anyone with FloPy can read
them back — which is how you pick up a model somebody else built.

In [ ]:
for f in sorted(WS.iterdir()):
    if f.suffix not in {".hds", ".cbc", ".list"}:
        print(f"  {f.name:18s} {f.stat().st_size:8,d} bytes")

In [ ]:
print((WS / "layered.nam").read_text())

The `.nam` file is the index: one line per package, giving the file unit number
and path. **That is the file you point `load()` at.**

In [ ]:
inherited = flopy.modflow.Modflow.load(
    "layered.nam", model_ws=str(WS), exe_name=str(MF_EXE),
    verbose=False, check=False,
)

print(f"packages found: {inherited.get_package_list()}")
print(f"grid: {inherited.dis.nlay} layers, {inherited.dis.nrow} rows, "
      f"{inherited.dis.ncol} columns")
print(f"stress periods: {inherited.dis.nper}")
print(f"layer bottoms: {inherited.dis.botm.array[:, 0, 0]}")

> **`check=False` is deliberate here.** FloPy's loader runs a consistency check by
> default, and on real inherited models it reports a wall of warnings about things
> that are unconventional but working. Turn it off to load, then run
> `inherited.check()` when you actually want to read them.

### YOUR TURN 4

Before changing anything, find out what you have inherited. **Never modify a
model you have not inspected.**

- `n_river_cells` — how many cells the `RIV` package acts on
- `well_rate` — the pumping rate in the `WEL` package
- `clay_hk` — horizontal conductivity of layer 1

The stress-period data lives in `inherited.riv.stress_period_data[0]`, a numpy
record array with named fields; conductivity is in
`inherited.lpf.hk.array[layer]`.

In [ ]:
# YOUR TURN
n_river_cells = ...
well_rate = ...
clay_hk = ...

In [ ]:
# CHECK
assert n_river_cells == NROW, f"expected {NROW} river cells, got {n_river_cells}"
assert abs(well_rate - PUMP) < 1e-6, f"got {well_rate}"
assert abs(clay_hk - 0.05) < 1e-9, f"got {clay_hk}"
print(f"river cells    {n_river_cells}")
print(f"well rate      {well_rate:,.0f} m3/d")
print(f"clay hk        {clay_hk} m/d")
print("Correct — now you know what you're changing.")

---

## Part 5 — Change one thing and compare

The clay conductivity was a guess. **Everything in a model that came from a guess
should be tested by changing it.**

In [ ]:
WS_ALT = RUN / "week14_leakier"
WS_ALT.mkdir(parents=True, exist_ok=True)

alt = flopy.modflow.Modflow.load(
    "layered.nam", model_ws=str(WS), exe_name=str(MF_EXE),
    verbose=False, check=False,
)

# a leakier clay: 20x the vertical conductivity
new_hk = alt.lpf.hk.array.copy()
new_vka = alt.lpf.vka.array.copy()
new_vka[1] *= 20.0

alt.lpf.hk = new_hk
alt.lpf.vka = new_vka

# point the model at a NEW directory so we don't overwrite the original
alt.change_model_ws(str(WS_ALT))
alt.write_input()
ok_alt, buff_alt = alt.run_model(silent=True, report=True)
assert ok_alt, "\n".join(buff_alt[-15:])

head_alt = flopy.utils.HeadFile(str(WS_ALT / "layered.hds")).get_data()
print(f"converged, discrepancy {discrepancy(WS_ALT / 'layered.list'):+.3f} %")
print(f"clay vka: {alt.lpf.vka.array[1, 0, 0]:.3f} m/d "
      f"(was {vka[1, 0, 0]:.3f})")

> **`change_model_ws()` before `write_input()`.** Without it you overwrite the
> original model with your modified one, and there is no undo. Every comparison
> you run should write to its own directory — that is the whole reason the
> comparison is possible.

### YOUR TURN 5

Compare the two runs.

- `diff_upper` — change in layer 0 head at the wellfield (leakier minus original)
- `diff_lower` — the same for layer 2
- `new_vertical_difference` — head difference across the clay in the leakier model

In [ ]:
# YOUR TURN
diff_upper = ...
diff_lower = ...
new_vertical_difference = ...

In [ ]:
# CHECK
assert new_vertical_difference < vertical_difference, (
    "a leakier clay should sustain LESS head difference, not more"
)
print(f"head difference across the clay:")
print(f"  original clay  {vertical_difference:7.2f} m")
print(f"  leakier clay   {new_vertical_difference:7.2f} m")
print(f"\nhead change at the wellfield:")
print(f"  layer 0 (upper) {diff_upper:+7.2f} m")
print(f"  layer 2 (lower) {diff_lower:+7.2f} m")
print("Correct.")

In [ ]:
alt_terms = budget_terms(WS_ALT / "layered.list")

print(f"{'term':18s} {'original':>13s} {'leakier clay':>14s}")
for name in sorted(set(terms) | set(alt_terms)):
    a = terms.get(name, {}).get("in", 0) - terms.get(name, {}).get("out", 0)
    b = alt_terms.get(name, {}).get("in", 0) - alt_terms.get(name, {}).get("out", 0)
    print(f"{name:18s} {a:13,.1f} {b:14,.1f}")

### YOUR TURN 6

The clay conductivity is one number that nobody measured, and it changes the
answer. Quantify how much.

- `river_original` — net river contribution in the original model
- `river_leakier` — net river contribution with the leakier clay
- `pct_change` — the change as a percentage of the original

In [ ]:
# YOUR TURN
river_original = ...
river_leakier = ...
pct_change = ...

In [ ]:
# CHECK
assert abs(river_original) > 1.0, "the river should be contributing something"
assert abs(pct_change) > 1.0, (
    "a 20x change in clay conductivity should move the river term by more than 1%"
)
print(f"net river contribution, original clay  {river_original:10,.1f} m3/d")
print(f"net river contribution, leakier clay   {river_leakier:10,.1f} m3/d")
print(f"change                                 {pct_change:+9.1f} %")
print("Correct.")

**That is the finding, and it is the point of the whole module.**

A parameter nobody measured — the vertical conductivity of a clay layer, known to
maybe a factor of ten — changes how much of the well's water comes out of the
river by a material amount. And the river is where the riparian vegetation is,
and where the downstream water right is.

Both models converged. Both closed their budgets. Both are defensible. **They
disagree about the thing the model was built to answer.**

The honest way to report this is not to pick one. It is to say: *under a clay
conductivity of 0.005 m/d the well captures X from the river; at 0.1 m/d it
captures Y; we do not know the clay conductivity to better than that range.*

A single model run is a hypothesis. **A pair of runs that bracket what you don't
know is a result.** That is what Project 3 asks you to produce.

---

## A note on real models

The model you just loaded had seven packages and 96 kB of input. A real regional
model is a different scale of object.

The **Pinal AMA model** used in this course's Project 3 has around 400 MB of
input files: 212 MB of evapotranspiration arrays, 100 MB of recharge, 12 MB of
well records, and a multi-node-well package describing individual pumps. It is
too large to keep in this repository, which is why it is distributed separately.

What does *not* change at that scale:

- `Modflow.load()` still reads it
- `change_model_ws()` before `write_input()` still saves you from overwriting it
- the `.list` file budget is still the first thing to read
- a run that converges is still not a run that is right

What does change is the time per run — minutes rather than seconds — which makes
the discipline of changing **one thing at a time** and keeping notes stop being
good practice and start being the only workable method.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 12 — Applying fluxes to a MODFLOW model**, Wednesday 12/2 at 11:59pm.

## Next week

Reading tracebacks, debugging, and getting code out of notebooks and into
modules — with five weeks of MODFLOW behind you to supply the bugs.

## Stuck?

- `Modflow.load()` raising on a package it doesn't recognise: pass
  `load_only=["DIS", "BAS6", "LPF"]` to read just the parts you need.
- Modifying `inherited.lpf.hk` in place sometimes doesn't take. Copy the array,
  change the copy, assign it back — as in Part 5.
- If your "modified" run gives identical results, you probably forgot
  `write_input()` after the change, so it ran the old files.
- If it overwrote your original, you forgot `change_model_ws()`. There is no undo;
  re-run Part 3.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.